# Modül 01 — Lineer Cebir | İnteraktif Keşif

Bu notebook, modülün 7 `.py` dosyasının **görsel ve interaktif özetidir**. Her bölüm:

1. Bir kavram için kısa Türkçe hatırlatma,
2. Canlı çalışan kod,
3. Bir görselleştirme.

`.py` dosyaları teoriyi taşır; bu notebook **gözle görmek için**. Sonda 5 "kendin dene" alıştırması var. Hücreleri sırayla çalıştır.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

# Tüm bölümlerde kullanılacak iki yardımcı: birim kare köşeleri ve birim çember.
def birim_kare():
    """Birim karenin 4 köşesi + kapanış noktası (fill için kapalı poligon)."""
    return np.array([[0, 1, 1, 0, 0],
                     [0, 0, 1, 1, 0]], dtype=float)

def birim_cember(N: int = 200):
    theta = np.linspace(0, 2 * np.pi, N)
    return np.stack([np.cos(theta), np.sin(theta)])


## 1 · Vektör geometrisi (`01_vektorler_geometri.py`)

Vektör = uzayda bir ok. **Toplam**: paralelogram kuralı. **İç çarpım**: aynı yöne ne kadar bakıyorlar. **Projeksiyon**: birinin diğerine "gölgesi".

Attention'da `Q[i] · K[j]` skoru tam olarak bu projeksiyon mantığı.

In [ ]:
u = np.array([3.0, 1.0])
v = np.array([1.0, 2.0])

fig, ax = plt.subplots(figsize=(6, 6))
ax.quiver(0, 0, *u, angles='xy', scale_units='xy', scale=1, color='C0', label='u')
ax.quiver(0, 0, *v, angles='xy', scale_units='xy', scale=1, color='C1', label='v')
ax.quiver(0, 0, *(u + v), angles='xy', scale_units='xy', scale=1, color='C2', label='u + v')
# Paralelogram kapanışı için u ucundan v vektörünü, v ucundan u vektörünü tekrar çiz.
ax.quiver(*u, *v, angles='xy', scale_units='xy', scale=1, color='C1', alpha=0.3)
ax.quiver(*v, *u, angles='xy', scale_units='xy', scale=1, color='C0', alpha=0.3)

ax.set_xlim(-1, 5); ax.set_ylim(-1, 5)
ax.set_aspect('equal'); ax.grid(True); ax.legend()
ax.set_title('Vektör toplama: paralelogram kuralı')
plt.show()

# İç çarpım → kosinüs benzerliği
cos_uv = np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))
print(f'cos(angle(u, v)) = {cos_uv:.3f}  (1.0 → aynı yön, 0 → dik, -1 → zıt)')


## 2 · Matris = Lineer Dönüşüm (`02_matris_lineer_donusum.py`)

Bir matrisin **sütunları**, standart taban vektörlerin gittiği yerdir. Birim kareyi farklı matrislerle dönüştürüp ne olduğunu gözleyelim.

In [ ]:
donusumler = {
    'Identity':         np.eye(2),
    'Ölçekle (2×, 0.5×)': np.array([[2.0, 0.0], [0.0, 0.5]]),
    'Rotasyon (45°)':    np.array([[np.cos(np.pi/4), -np.sin(np.pi/4)],
                                    [np.sin(np.pi/4),  np.cos(np.pi/4)]]),
    'Shear (x)':         np.array([[1.0, 1.0], [0.0, 1.0]]),
    'Aynala (x ekseni)': np.array([[1.0, 0.0], [0.0, -1.0]]),
}

kare = birim_kare()
fig, axs = plt.subplots(1, len(donusumler), figsize=(4 * len(donusumler), 4))
for ax, (ad, A) in zip(axs, donusumler.items()):
    yeni = A @ kare
    ax.fill(kare[0], kare[1], alpha=0.3, color='gray', label='orijinal')
    ax.fill(yeni[0], yeni[1], alpha=0.5, color='C0', label='dönüşmüş')
    ax.set_xlim(-2, 3); ax.set_ylim(-2, 3)
    ax.set_aspect('equal'); ax.grid(True)
    ax.set_title(f'{ad}\nsütunlar = {A[:, 0]}, {A[:, 1]}')
axs[0].legend(loc='upper right')
plt.tight_layout(); plt.show()


## 3 · Çarpım = Kompozisyon (`03_matris_carpim_kompozisyon.py`)

`(AB) x = A (B x)` — yani `AB` çarpımı "önce B, sonra A" anlamına gelir. Bu yüzden `AB ≠ BA` genelde. Aynı iki dönüşüm farklı sırada uygulandığında **görünüşü gerçekten değişir**.

In [ ]:
R = np.array([[np.cos(np.pi/4), -np.sin(np.pi/4)],
              [np.sin(np.pi/4),  np.cos(np.pi/4)]])
Sh = np.array([[1.0, 1.0], [0.0, 1.0]])

kare = birim_kare()
fig, axs = plt.subplots(1, 3, figsize=(13, 4))

axs[0].fill(kare[0], kare[1], alpha=0.4)
axs[0].set_title('Orijinal birim kare')

# Önce R, sonra Sh: matris (Sh @ R)
sira1 = (Sh @ R) @ kare
axs[1].fill(sira1[0], sira1[1], alpha=0.5, color='C3')
axs[1].set_title('Sh @ R  (önce R, sonra Sh)')

# Önce Sh, sonra R: matris (R @ Sh)
sira2 = (R @ Sh) @ kare
axs[2].fill(sira2[0], sira2[1], alpha=0.5, color='C0')
axs[2].set_title('R @ Sh  (önce Sh, sonra R)')

for ax in axs:
    ax.set_xlim(-3, 3); ax.set_ylim(-1, 3)
    ax.set_aspect('equal'); ax.grid(True)
plt.tight_layout(); plt.show()

print('Sh @ R =\n', Sh @ R)
print('R @ Sh =\n', R @ Sh)
print('Aynı matris mi?', np.allclose(Sh @ R, R @ Sh))


## 4 · Determinant = Alan Çarpanı (`04_determinant_ters.py`)

`|det A|` = matris birim kareyi kaç kat alana ölçeklendiriyor. İşareti = yönelim korunmuş mu (`+`), tersine dönmüş mü (`−`). `det = 0` ise dönüşüm bir doğruya çöker; bilgi kaybolur, geri alınamaz.

In [ ]:
matrisler = {
    'det = 2':   np.array([[2.0, 0.0], [0.0, 1.0]]),
    'det = 1':   np.array([[np.cos(np.pi/6), -np.sin(np.pi/6)],
                          [np.sin(np.pi/6),  np.cos(np.pi/6)]]),
    'det = 0.5': np.array([[0.5, 0.0], [0.0, 1.0]]),
    'det = -1':  np.array([[1.0, 0.0], [0.0, -1.0]]),
    'det = 0':   np.array([[1.0, 2.0], [2.0, 4.0]]),
}

kare = birim_kare()
fig, axs = plt.subplots(1, 5, figsize=(20, 4))
for ax, (etiket, A) in zip(axs, matrisler.items()):
    yeni = A @ kare
    ax.fill(kare[0], kare[1], alpha=0.3, color='gray')
    ax.fill(yeni[0], yeni[1], alpha=0.5, color='C0')
    gerçek = np.linalg.det(A)
    ax.set_title(f'{etiket}\n(numpy: {gerçek:+.2f})')
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
    ax.set_aspect('equal'); ax.grid(True)
plt.tight_layout(); plt.show()


## 5 · Özdeğer / Özvektör (`05_ozdeger_ozvektor.py`)

Bir matrisin **özvektörleri** o yönü değiştirmeden tutar, yalnızca **özdeğer** kadar uzatıp kısaltır. Yani matrisin doğal eksenleri buradadır. Birim çemberi dönüştürürsek, **özvektör yönleri elipsin uzun/kısa eksenleriyle hizalanır** (simetrik matrislerde).

In [ ]:
# Simetrik matris seçtik ki özvektörler dik olsun ve görselleştirme net olsun.
A = np.array([[3.0, 1.0],
              [1.0, 2.0]])

lam, V = np.linalg.eigh(A)  # eigh sıralı döner; simetrik için tercih edilir.
print('Özdeğerler  λ =', lam)
print('Özvektörler V =\n', V)

cember = birim_cember()
elips = A @ cember

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(cember[0], cember[1], 'b-', alpha=0.5, label='birim çember')
ax.plot(elips[0], elips[1], 'r-', label='A · çember = elips')
for i in range(2):
    v_yön = V[:, i]
    # Siyah ok: orijinal yön; yeşil ok: A altında uzatılmış hâl.
    ax.quiver(0, 0, *v_yön, color='black', angles='xy', scale_units='xy', scale=1, alpha=0.6,
              label='özvektör' if i == 0 else None)
    ax.quiver(0, 0, *(lam[i] * v_yön), color='green', angles='xy', scale_units='xy', scale=1,
              label='λ · özvektör' if i == 0 else None)

ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)
ax.set_aspect('equal'); ax.grid(True); ax.legend()
ax.set_title('Özvektörler dönüşüm sonrası yönlerini korur')
plt.show()


## 6 · SVD = "Döndür → Ölçekle → Döndür" (`06_svd.py`)

Her matris `A = U Σ Vᵀ` olarak yazılır. Üç adımı tek tek uygulayıp birim çemberin elipse nasıl dönüştüğünü gözleyelim.

In [ ]:
A = np.array([[3.0, 1.0],
              [1.0, 2.0]])

U, s, Vt = np.linalg.svd(A)
print('σ =', s)

cember = birim_cember()
adim_0 = cember                # birim çember
adim_1 = Vt @ cember           # Vᵀ döndürdü
adim_2 = np.diag(s) @ adim_1   # Σ eksenleri gerdi → elips, ama koordinat eksenlerinde
adim_3 = U @ adim_2            # U yeniden döndürdü → A · çember

fig, axs = plt.subplots(1, 4, figsize=(20, 5))
for ax, adim, baslik in zip(
    axs,
    [adim_0, adim_1, adim_2, adim_3],
    ['1) birim çember', '2) Vᵀ uygulandı', '3) Σ uygulandı', '4) U uygulandı  (= A·çember)'],
):
    ax.plot(adim[0], adim[1], 'b-')
    ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
    ax.set_aspect('equal'); ax.grid(True)
    ax.set_title(baslik)
plt.tight_layout(); plt.show()

# Doğrulama: dört adımlı sonuç ≈ A @ çember.
print('U Σ Vᵀ ile A·çember farkı (Fro):', np.linalg.norm(adim_3 - A @ cember))


## 7 · Düşük-Rank Yaklaşım = Görüntü Sıkıştırma (`07_dusuk_rank_yaklasim.py`)

İlk `k` tekil değeri tutup geri kalanı atmak → en iyi rank-k yaklaşım (Eckart-Young). Sentetik bir 64×64 görüntü üzerinde `k` arttıkça kalitenin nasıl yükseldiğini izleyelim.

In [ ]:
def gorsel_uret(N: int = 64) -> np.ndarray:
    x = np.linspace(0, 4 * np.pi, N)
    y = np.linspace(0, 4 * np.pi, N)
    X, Y = np.meshgrid(x, y)
    img = np.sin(X) * np.cos(Y) + 0.3 * np.sin(2 * X + Y)
    img[10:30, 10:30] += 1.5
    img += 0.05 * np.random.default_rng(42).standard_normal(img.shape)
    return img

def yaklaslik_k(A: np.ndarray, k: int) -> np.ndarray:
    U, s, Vt = np.linalg.svd(A, full_matrices=False)
    return U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]

img = gorsel_uret()
ks = [1, 2, 4, 8, 16, 64]

fig, axs = plt.subplots(1, len(ks), figsize=(3 * len(ks), 3.5))
for ax, k in zip(axs, ks):
    img_k = yaklaslik_k(img, k)
    ax.imshow(img_k, cmap='gray')
    hata = np.linalg.norm(img - img_k, ord='fro')
    ax.set_title(f'k = {k}\n‖A − Aₖ‖ = {hata:.2f}')
    ax.axis('off')
plt.tight_layout(); plt.show()

# Tekil değer spektrumu — "enerji nasıl dağılmış?"
_, s, _ = np.linalg.svd(img, full_matrices=False)
kumulatif = np.cumsum(s ** 2) / np.sum(s ** 2)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].semilogy(s, 'o-'); axs[0].set_title('Tekil değerler (log y)')
axs[0].set_xlabel('i'); axs[0].set_ylabel('σᵢ'); axs[0].grid(True)
axs[1].plot(kumulatif, 'o-'); axs[1].axhline(0.99, color='r', ls='--', label='99% enerji')
axs[1].set_title('Kümülatif açıklanan varyans'); axs[1].set_xlabel('k')
axs[1].grid(True); axs[1].legend()
plt.tight_layout(); plt.show()

esik = int(np.argmax(kumulatif > 0.99)) + 1
print(f'Görüntünün enerjisinin %99\'unu taşıyan en küçük k = {esik}')


## 8 · Kendin Dene

Aşağıdaki 5 alıştırmayı boş hücrelerde çöz. Hepsi notebook'ta gördüğün araçlarla çözülebilir.

1. **2×2 dönüşüm okuma**: `A = [[2, 1], [0, 2]]` matrisinin determinantını, özdeğerlerini ve özvektörlerini hesapla. Birim kareyi dönüştür ve sonucu çiz. Bu matris ne yapıyor — ölçekle mi, döndür mü, shear mı?

2. **Rotasyon kompozisyonu**: `R(30°) @ R(20°)` ile `R(50°)` aynı matris mi? Sayısal olarak doğrula. (İpucu: hepsinin determinantı 1 olmalı ve sütunları birim vektör olmalı.)

3. **SVD'de işaret belirsizliği**: `A = [[3, 1], [1, 2]]` için SVD'yi al; bir de `−A` için SVD'yi al. `U`, `Σ`, `Vᵀ` arasında ne değişti, ne değişmedi?

4. **Sıkıştırma bütçesi**: Yukarıdaki 64×64 görüntünün enerjisinin %95'i için kaç `k` yetiyor? Parametre tasarrufu yaklaşık kaç kat? (İpucu: `64×64 = 4096` toplam piksel; `k·(64 + 64 + 1)` ile karşılaştır.)

5. **Markov yakınsama**: Modül 05'teki hava durumu geçiş matrisini kur. Başlangıçta `[1, 0, 0]` (kesin güneşli) varsayıp 1, 5, 50 adım sonra olasılık dağılımını yazdır. Durağan dağılıma kaç adımda yakınsıyor?

In [ ]:
# Çözüm alanın — buraya yaz.
